# Sand and Dust Storm (SDS) Analysis — Dubai
Methodology based on: *Remote sensing-based sand and dust storm analysis using MODIS data* (2022)
https://www.tandfonline.com/doi/full/10.1080/22797254.2022.2093278

In [1]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='sds-analysis-493219')

print("Connected to Google Earth Engine!")

Connected to Google Earth Engine!


## 1. Area of Interest: Dubai

In [2]:

# ======================================================
# Define ROI: Dubai using geoBoundaries ADM2
# ======================================================
adm2 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")

dubaiFC = (
    adm2
    .filter(ee.Filter.eq('shapeGroup', 'ARE'))  # UAE ISO3 = ARE
    .filter(ee.Filter.eq('shapeName', 'Dubai'))
)

roi = dubaiFC.geometry()
print('Dubai features found:', dubaiFC.size().getInfo())


Dubai features found: 1


### Map: Area of Interest

In [3]:

# ======================================================
# Display the Area of Interest
# ======================================================
aoi_map = geemap.Map(basemap="HYBRID")
aoi_map.centerObject(roi, 9)

aoi_map.addLayer(
    dubaiFC,
    {'color': '#FF6600', 'fillColor': '#FF660033'},
    'Dubai AOI'
)

aoi_map


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

## 2. Load Datasets & Define Helpers

In [4]:

# ======================================================
# Load MODIS datasets
# ======================================================
mod09 = ee.ImageCollection("MODIS/061/MOD09GA")

# ======================================================
# QA masking + reflectance scaling
# Spectral indices are now computed separately via compute_all_indices.py
# ======================================================
def scale_and_mask_MOD09GA(img):
    """Scale SR bands to 0.0001 reflectance. Cloud masking deferred to pipeline."""
    b = img.select([
        "sur_refl_b01", "sur_refl_b02", "sur_refl_b03", "sur_refl_b04",
        "sur_refl_b05", "sur_refl_b06", "sur_refl_b07"
    ]).multiply(0.0001)

    # NOTE: Cloud screening is intentionally omitted here.
    # Dust pixels can trigger cloud QA flags — we keep them and let the
    # multi-index thresholding pipeline (Section 7) separate dust from cloud.
    return (
        img.addBands(b, None, True)
           .copyProperties(img, img.propertyNames())
    )



### 2.5 Load Auxillary Datasets

In [5]:
# ==============================================================================
# AUXILIARY DATASETS — loaded once at module import
# ==============================================================================

# SRTM DEM: required for SI class (elevation > 2500 m).
# Dubai max elevation ≈ 300 m — SI pixels will not occur unless ROI is expanded
# to include the Hajar Mountains (Oman/UAE, up to ~3000 m).
srtm_band = (
    ee.Image('USGS/SRTMGL1_003')
    .select('elevation')
    .reproject(crs='EPSG:4326', scale=500)
    .clip(roi)
    .rename('DEM')
)

# MCD12Q1 land cover — IGBP Type 1. Water body = class 17.
# Year 2020 used as default; adjust if analysing a different year.
# Used for WD class (dust over water bodies).
lct_band = (
    ee.ImageCollection('MODIS/061/MCD12Q1')
    .filter(ee.Filter.calendarRange(2020, 2020, 'year'))
    .first()
    .select('LC_Type1')
    .reproject(crs='EPSG:4326', scale=500)
    .clip(roi)
    .rename('LCT')
)

IGBP_WATER = 17   # IGBP water body class code


# ==============================================================================
# FUNCTION 1 — ATTACH AUXILIARY DATA
# ==============================================================================

def attach_auxiliary_data(image, lct_year=2020):
    """
    Attach SRTM DEM and MCD12Q1 land cover type as bands to the image.

    Both datasets are reprojected to match the MOD09GA 500 m pixel grid.
    DEM uses bilinear resampling (continuous elevation values).
    LCT uses nearest-neighbour (preserves integer class codes).

    These auxiliary bands are required by various Table 3 criteria:
        SI  → DEM > 2500 m   (separates high-altitude snow from thick cloud)
        WD  → LCT == 17      (dust over IGBP water body land cover)

    Parameters
    ----------
    image : ee.Image
        Image with reflectance and index bands already attached.
    lct_year : int
        Year of MCD12Q1 land cover to use. Defaults to 2020.

    Returns
    -------
    ee.Image
        Input image with 'DEM' and 'LCT' bands added.
    """

    modis_proj = image.select('sur_refl_b01').projection()
# DEM — reproject only, no .resample() call needed
    dem_500m = (
        srtm_band
        .reproject(crs=modis_proj, scale=500)
        .rename('DEM')
    )

    # LCT — reproject only, GEE uses nearest neighbour by default
    # for integer/categorical images when no resample is specified
    lct_500m = (
        lct_band
        .reproject(crs=modis_proj, scale=500)
        .rename('LCT')
    )

    return image.addBands([dem_500m, lct_500m])

## 3. Build Image Collection & Select Analysis Date

In [6]:

import importlib, sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import compute_all_indices as _ci_mod
importlib.reload(_ci_mod)
from compute_all_indices import compute_all_indices

# ======================================================
# Study period
# ======================================================
start = "2022-01-01"
end   = "2022-12-31"

# Build scaled collection — no indices yet (applied per-image below)
ic_raw = (
    mod09
    .filterDate(start, end)
    .filterBounds(roi)
    .map(scale_and_mask_MOD09GA)
)

# ======================================================
# Select a dust event date (April 2022 — known active SDS season)
# ======================================================
date = ee.Date('2022-05-23') # May 23, 2022 # April 18, 2022

day_col = ic_raw.filterDate(date, date.advance(1, 'day'))
print('Images found for 2022-05-23:', day_col.size().getInfo())

img_raw = ee.Image(day_col.sort('system:time_start').first()).clip(roi)

# img: scaled reflectance + spectral indices — used for Section 4 maps
img = compute_all_indices(img_raw)
print('Index bands added:', ['NDDI','NDVI','NDWI','NDSI','MBWI','MBSCI'])


Images found for 2022-05-23: 1
Index bands added: ['NDDI', 'NDVI', 'NDWI', 'NDSI', 'MBWI', 'MBSCI']


## 4. Spectral Indices Maps

Three most important indices for SDS detection:

| Index | Formula | Role in SDS |
|-------|---------|-------------|
| **NDDI** | (SWIR3 − Blue) / (SWIR3 + Blue) | Primary dust aerosol detector — high values = active dust |
| **MBSCI** | (Green + Red) / (SWIR2 + SWIR3) | Identifies bare/sandy surfaces (sediment source areas) |
| **NDVI** | (NIR − Red) / (NIR + Red) | Vegetation cover — low values = exposed dust source |


In [7]:

# ======================================================
# Map 1: NDDI — Normalized Difference Dust Index
# Higher values indicate active dust/sand storm presence
# ======================================================
nddi_map = geemap.Map(basemap="HYBRID")
nddi_map.centerObject(roi, 9)

nddi_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
nddi_map.addLayer(
    img.select('NDDI'),
    {
        'min': -0.3,
        'max':  0.6,
        'palette': ['#2166ac', '#92c5de', '#f7f7f7', '#fddbc7', '#d6604d', '#b2182b']
    },
    'NDDI (Dust Index)'
)

nddi_map.add_colorbar(
    vis_params={'min': -0.3, 'max': 0.6,
                'palette': ['#2166ac', '#92c5de', '#f7f7f7', '#fddbc7', '#d6604d', '#b2182b']},
    label='NDDI',
    orientation='horizontal',
    layer_name='NDDI (Dust Index)'
)

print("NDDI — higher (red) = active dust/sand aerosol")
nddi_map



NDDI — higher (red) = active dust/sand aerosol


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [8]:

# ======================================================
# Map 2: MBSCI — Modified Bare Soil Composition Index
# Identifies bare sandy/dusty surfaces (sediment source areas)
# ======================================================
mbsci_map = geemap.Map(basemap="HYBRID")
mbsci_map.centerObject(roi, 9)

mbsci_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
mbsci_map.addLayer(
    img.select('MBSCI'),
    {
        'min':  0.3,
        'max':  1.5,
        'palette': ['#4d9221', '#a1d76a', '#f7f7f7', '#e9a3c9', '#c51b7d']
    },
    'MBSCI (Bare Soil)'
)
mbsci_map.add_colorbar(
    vis_params={'min': 0.3, 'max': 1.5,
                'palette': ['#4d9221', '#a1d76a', '#f7f7f7', '#e9a3c9', '#c51b7d']},
    label='MBSCI',
    orientation='horizontal',
    layer_name='MBSCI (Bare Soil)'
)

print("MBSCI — higher (pink/purple) = more exposed bare/sandy soil")
mbsci_map


MBSCI — higher (pink/purple) = more exposed bare/sandy soil


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [9]:

# ======================================================
# Map 3: NDVI — Normalized Difference Vegetation Index
# Low NDVI = exposed arid surface = higher SDS susceptibility
# ======================================================
ndvi_map = geemap.Map(basemap="HYBRID")
ndvi_map.centerObject(roi, 9)

ndvi_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
ndvi_map.addLayer(
    img.select('NDVI'),
    {
        'min': -0.1,
        'max':  0.5,
        'palette': ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e']
    },
    'NDVI (Vegetation)'
)
ndvi_map.add_colorbar(
    vis_params={'min': -0.1, 'max': 0.5,
                'palette': ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e']},
    label='NDVI',
    orientation='horizontal',
    layer_name='NDVI (Vegetation)'
)

print("NDVI — brown/low = arid exposed surface (high SDS risk); green/high = vegetated")
ndvi_map


NDVI — brown/low = arid exposed surface (high SDS risk); green/high = vegetated


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

## 5. Enhanced Dust Index (EDI)

### 5.1 Enhanced Dust Index with Dust Fraction α (EDI-α)

Implemented in [`edi_with_alpha.py`](edi_with_alpha.py) following Wang et al. (2022) / Han et al. (2013).

This method first runs **Linear Spectral Unmixing (LSU)**
to estimate α — the fraction of each pixel that is "pure dust" — then uses α to asymmetrically
weight the SWIR and Blue reflectance terms:

$$\text{EDI} = \frac{\alpha \cdot \rho_{2.13} - (1-\alpha) \cdot \rho_{0.469}}{\alpha \cdot \rho_{2.13} + (1-\alpha) \cdot \rho_{0.469}}$$

This suppresses false positives over bright bare desert soil, which can mimic dust in simple
band-ratio indices but gets low α from LSU because its spectral shape better matches the
bare-soil endmember.

In [10]:

# Import EDI-α module (see edi_with_alpha.py for full methodology comments)
import importlib, sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import edi_with_alpha as edi_alpha_module
importlib.reload(edi_alpha_module)

from edi_with_alpha import compute_dust_fraction, compute_edi_alpha, classify_edi_alpha

# Step 1: Linear Spectral Unmixing → adds 'alpha_dust' band
img_with_alpha = compute_dust_fraction(img)

# Step 2: α-weighted EDI → adds 'EDI_alpha' band
img_edi_alpha  = compute_edi_alpha(img_with_alpha)

# Step 3: 3-class dust likelihood map
edi_alpha_band  = img_edi_alpha.select('EDI_alpha')
edi_alpha_class = classify_edi_alpha(edi_alpha_band)

print("Bands available:", img_edi_alpha.bandNames().getInfo())


Bands available: ['num_observations_1km', 'state_1km', 'SensorZenith', 'SensorAzimuth', 'Range', 'SolarZenith', 'SolarAzimuth', 'gflags', 'orbit_pnt', 'granule_pnt', 'num_observations_500m', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05', 'sur_refl_b06', 'sur_refl_b07', 'QC_500m', 'obscov_500m', 'iobs_res', 'q_scan', 'NDDI', 'NDVI', 'NDWI', 'NDSI', 'MBWI', 'MBSCI', 'alpha_dust', 'alpha_cloud', 'alpha_water', 'EDI_alpha']


In [11]:

# ======================================================
# Map: EDI-α (continuous)
# ======================================================
edi_alpha_vis = {
    'min': -0.2,
    'max':  0.4,
    'palette': [
        '#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8',
        '#ffffbf', '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026'
    ]
}

edi_alpha_map = geemap.Map(basemap="HYBRID")
edi_alpha_map.centerObject(roi, 9)
edi_alpha_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')
edi_alpha_map.addLayer(edi_alpha_band, edi_alpha_vis, 'EDI-α (continuous)')

edi_alpha_map.add_colorbar(
    vis_params=edi_alpha_vis,
    label='EDI-α  (blue = non-dust   →   red = active dust)',
    orientation='horizontal'
)

edi_alpha_map


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [12]:

# ======================================================
# Map: EDI-α Classification (0 = non-dust, 1 = moderate, 2 = high)
# ======================================================
edi_class_vis = {
    'min': 0,
    'max': 2,
    'palette': ['#d9d9d9', '#fdae61', '#d73027']   # grey | orange | red
}

edi_class_map = geemap.Map(basemap="HYBRID")
edi_class_map.centerObject(roi, 9)
edi_class_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')
edi_class_map.addLayer(edi_alpha_class, edi_class_vis, 'EDI-α class')

edi_class_map.add_legend(
    title='EDI-α Dust Likelihood',
    legend_dict={
        '0 — Non-dust (EDI < 0.05)':        '#d9d9d9',
        '1 — Moderate dust (0.05 – 0.20)':  '#fdae61',
        '2 — High dust (EDI ≥ 0.20)':       '#d73027',
    }
)

edi_class_map


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

## 7. Multi-Index Threshold Labelling (Wang et al. 2022 — Table 3)

Implements the full AL-SVM pre-labelling pipeline from `multi_index_thresholding.py`:

| Step | Function | Script |
|------|----------|--------|
| 1 | `compute_all_indices()` | `compute_all_indices.py` |
| 2 | `compute_dust_fraction()` + `compute_edi_alpha()` | `edi_with_alpha.py` |
| 3 | `attach_auxiliary_data()` | `attach_auxiliary_data()` |
| 4 | `apply_multi_index_thresholds()` | `apply_multi_index_thresholds.py` |

Output: single-band `class_label` image (uint8, 0–11) — 11 surface/atmospheric classes
used as training samples for the downstream SVM classifier.

In [13]:

# ── Imports ───────────────────────────────────────────────────────────────────
import importlib, sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))

import edi_with_alpha as _ewa; importlib.reload(_ewa)
import compute_all_indices as _ci; importlib.reload(_ci)

from edi_with_alpha import compute_dust_fraction, compute_edi_alpha
from compute_all_indices import compute_all_indices


# ── label_image: full pipeline from scaled reflectance → class label ──────────
def get_indices(image):
    """
    Run the complete Wang et al. (2022) Table 3 labelling pipeline on a
    single scaled MOD09GA image.

    Steps:
        1. compute_all_indices    → NDDI, NDVI, NDWI, NDSI, MBWI, MBSCI
        2. compute_dust_fraction  → alpha_dust, alpha_cloud, alpha_water (LSU)
        3. compute_edi_alpha      → EDI_alpha (α-weighted dust index)
        4. attach_auxiliary_data  → DEM, LCT bands
        5. apply_multi_index_thresholds → class_label (0–11)

    Parameters
    ----------
    image : ee.Image
        Scaled MOD09GA image (sur_refl_b* bands, 0–1 range).

    Returns
    -------
    ee.Image
        Single-band 'class_label' image (uint8, values 0–11).
    """
    image = compute_all_indices(image)
    image = compute_dust_fraction(image)
    image = compute_edi_alpha(image)
    # image = attach_auxiliary_data(image)
    return image


# ── Apply to the analysis date ────────────────────────────────────────────────
# img_raw is the scaled reflectance-only image from Section 3
image_processed = get_indices(img_raw).clip(roi)

# print("class_label band:", class_label.bandNames().getInfo())
# print("Data type:", class_label.getInfo()['bands'][0]['data_type']['precision'])


In [14]:
print("class_label band:", image_processed.bandNames().getInfo())

class_label band: ['num_observations_1km', 'state_1km', 'SensorZenith', 'SensorAzimuth', 'Range', 'SolarZenith', 'SolarAzimuth', 'gflags', 'orbit_pnt', 'granule_pnt', 'num_observations_500m', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05', 'sur_refl_b06', 'sur_refl_b07', 'QC_500m', 'obscov_500m', 'iobs_res', 'q_scan', 'NDDI', 'NDVI', 'NDWI', 'NDSI', 'MBWI', 'MBSCI', 'alpha_dust', 'alpha_cloud', 'alpha_water', 'EDI_alpha']


In [15]:
image_prc = attach_auxiliary_data(image_processed)

In [16]:
srtm_band = image_prc.select("DEM")
srtm_vis = {
        'min':   0,
        'max': 300,       # Dubai max elevation ~300m — adjust if ROI expanded
        'palette': [
            '#f7fbff',    # near sea level — white/pale blue
            '#c6dbef',
            '#9ecae1',
            '#6baed6',
            '#3182bd',
            '#08519c',    # highest terrain — dark blue
        ]
    }

lc_band = image_prc.select("LCT")

lc_vis = {'min': 0, 'max': 300,
     'palette': ['#f7fbff','#c6dbef','#9ecae1','#6baed6','#3182bd','#08519c']}

### 7.1 Visualisation — Auxillary Data Map

In [17]:

# ======================================================
# Map: SRTM Map

# ======================================================
class_map = geemap.Map(basemap="HYBRID")
class_map.centerObject(roi, 9)

# class_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')
class_map.addLayer(srtm_band, srtm_vis, 'SRTM')

# class_map.add_legend(
#     title='Wang et al. 2022 — Table 3 Classes',
#     legend_dict=CLASS_LEGEND
# )

class_map


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [18]:
# ======================================================
### LC Map
# ======================================================
class_map = geemap.Map(basemap="HYBRID")
class_map.centerObject(roi, 9)

class_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')
class_map.addLayer(lc_band, lc_vis, 'LC')

# class_map.add_legend(
#     title='Wang et al. 2022 — Table 3 Classes',
#     legend_dict=CLASS_LEGEND
# )

class_map

Map(center=[24.94287090991497, 55.38817725705568], controls=(WidgetControl(options=['position', 'transparent_b…

## 7.2 Object classification using Multi Index Threshold 

In [19]:
import apply_multi_index_thresholds as _amt; importlib.reload(_amt)
from apply_multi_index_thresholds import (
    apply_multi_index_thresholds,
    VIS_CLASS, CLASS_LEGEND
)
image_class = apply_multi_index_thresholds(image_prc)

In [20]:
print("class_label band:", image_class.bandNames().getInfo())

class_label band: ['class_label']


In [21]:

# ======================================================
# Map: Class Map

# ======================================================
class_map = geemap.Map(basemap="HYBRID")
class_map.centerObject(roi, 9)

# class_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')
class_map.addLayer(image_class, VIS_CLASS, 'SRTM')

class_map.add_legend(
    title='Wang et al. 2022 — Table 3 Classes',
    legend_dict=CLASS_LEGEND
)

class_map

Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

# Visualization

In [24]:
image_prc.bandNames()

In [25]:
# ======================================================
# Visualise SDS pixels (TcD + TnD + WD) alongside
# MODIS true-colour image for the same date
# ======================================================

import ee
import geemap

# ── Class codes (must match your thresholding script) ─────────────────────────
CLASS_TCD = 4    # Thick Dust
CLASS_TND = 5    # Thin Dust
CLASS_WD  = 6    # Water Dust

# ── Build the SDS mask from your class_label image ────────────────────────────
# Merge the three dust classes into a single binary mask.
# Result: 1 = dust (any type), 0 = everything else
sds_mask = (
    image_class.eq(CLASS_TCD)
    .Or(image_class.eq(CLASS_TND))
    .Or(image_class.eq(CLASS_WD))
    .rename('SDS')
)

# ── Build the three-class dust image for colour-coded display ─────────────────
# Rather than a flat binary mask, show each dust type in its own colour
# so you can visually assess whether TcD and TnD are spatially sensible.
# Assign: TcD=1, TnD=2, WD=3, non-dust=0
dust_typed = (
    ee.Image(0)                                        # start: all zero
    .where(image_class.eq(CLASS_TND), 2)         # TnD first (lower priority)
    .where(image_class.eq(CLASS_WD),  3)         # WD
    .where(image_class.eq(CLASS_TCD), 1)         # TcD last (highest priority)
    .rename('dust_type')
    .clip(roi)
)

# ── MODIS true-colour composite ───────────────────────────────────────────────
# True colour = R:B1(Red)  G:B4(Green)  B:B3(Blue)
# This matches exactly how Wang et al. Figure 5 displays true-colour images.
# img is your compute_all_indices output image for the event date.
true_colour = image_prc.select(['sur_refl_b01', 'sur_refl_b04', 'sur_refl_b03']).clip(roi)

# ── Visualisation parameters ──────────────────────────────────────────────────

# True colour — stretch to typical MODIS surface reflectance range
vis_tc = {
    'min':  0.0,
    'max':  0.35,
    'gamma': 1.4    # slight gamma boost makes desert imagery more readable
}

# Dust types — transparent background, three distinct colours
# TcD = dark red (dense storm core)
# TnD = orange   (thin veil, periphery of storm)
# WD  = yellow   (dust over Gulf/Creek water)
vis_dust_typed = {
    'min':     0,
    'max':     3,
    'palette': [
        '00000000',   # 0 — non-dust: fully transparent
        'd73027',     # 1 — TcD Thick Dust:  dark red
        'fdae61',     # 2 — TnD Thin Dust:   orange
        'ffffbf',     # 3 — WD  Water Dust:  pale yellow
    ]
}

# Binary SDS mask — single bold colour for clean overview
vis_sds_binary = {
    'min':     0,
    'max':     1,
    'palette': ['00000000', 'd73027']   # transparent | red
}

# ── Count pixels per dust class for a quick sanity check ─────────────────────
def count_class(label_img, class_val, region, scale=500):
    """Return pixel count for a single class value within the region."""
    return (
        label_img.eq(class_val)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region,
            scale=scale,
            maxPixels=1e9
        )
        .values()
        .get(0)
    )

n_tcd = count_class(image_class, CLASS_TCD, roi)
n_tnd = count_class(image_class, CLASS_TND, roi)
n_wd  = count_class(image_class, CLASS_WD,  roi)

print(f"TcD pixels : {n_tcd.getInfo():>6}")
print(f"TnD pixels : {n_tnd.getInfo():>6}")
print(f"WD  pixels : {n_wd.getInfo():>6}")
print(f"Total SDS  : {ee.Number(n_tcd).add(n_tnd).add(n_wd).getInfo():>6}")

TcD pixels :      0
TnD pixels : 47.011764705882364
WD  pixels :      0
Total SDS  : 47.011764705882364


In [26]:
# ======================================================
# Map: Side-by-side true colour + SDS overlay
# ======================================================

Map = geemap.Map(center=[25.2048, 55.2708], zoom=9)
Map.add_basemap('HYBRID')

# Layer 1 — MODIS true colour (bottom)
Map.addLayer(
    true_colour,
    vis_tc,
    'MODIS True Colour (R:B1 G:B4 B:B3)',
    shown=True
)

# Layer 2 — Dust type (colour-coded, semi-transparent)
# Toggle this off to see true colour cleanly beneath
Map.addLayer(
    dust_typed,
    vis_dust_typed,
    'Dust classes: TcD / TnD / WD',
    shown=True,
    opacity=0.75
)

# Layer 3 — Binary SDS mask (flat red, useful for clean overview)
Map.addLayer(
    sds_mask,
    vis_sds_binary,
    'SDS binary mask (TcD+TnD+WD)',
    shown=False    # hidden by default — toggle on to compare
)

# Layer 4 — Full 11-class label image (for context)
CLASS_PALETTE = [
    'ffffff',   # 0  Unclassified
    '4575b4',   # 1  TcC  Thick Cloud
    '74add1',   # 2  Cs   Cirrostratus
    'abd9e9',   # 3  TnC  Thin Cloud
    'd73027',   # 4  TcD  Thick Dust
    'fdae61',   # 5  TnD  Thin Dust
    'ffffbf',   # 6  WD   Water Dust
    'f4a582',   # 7  Lb   Dry Lakebed
    '313695',   # 8  WB   Water Bodies
    'c8a96e',   # 9  BL   Bare Land
    '1a9641',   # 10 VL   Vegetated Land
    'e0f3f8',   # 11 SI   Snow and Ice
]
Map.addLayer(
    image_class.clip(roi),
    {'min': 0, 'max': 11, 'palette': CLASS_PALETTE},
    'All 11 classes (reference)',
    shown=False
)

# ROI boundary
Map.addLayer(
    ee.Image().paint(roi, 1, 2),
    {'palette': ['ffffff']},
    'Dubai ROI boundary'
)

# ── Legend ────────────────────────────────────────────────────────────────────
Map.add_legend(
    title='Dust classes',
    legend_dict={
        'TcD — Thick Dust':  '#d73027',
        'TnD — Thin Dust':   '#fdae61',
        'WD  — Water Dust':  '#ffffbf',
        'Non-dust':          '#cccccc',
    }
)

Map

Map(center=[25.2048, 55.2708], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…